# GSB 5544 — Topic 7.1: Association Rules  
*Who gets played together? Fill each `____` blank as you work; the ✅ checks ask for a sentence or two.*

## The next 20 minutes

| | Question | New tools | Where it lands in PA 7.1 |
|---|---|---|---|
| **1. Baskets** | What is a transaction, and how does a list of lists become a table of `True`/`False`? | `explode`, `crosstab` | parts 1 – 3 |
| **2. Support** | How common is an item — or a pair of items? | `.mean()` on booleans, `.all(axis=1)` | parts 4 – 5, 9 |
| **3. Confidence & lift** | If a playlist has A, how likely is B — and is that more than chance? | two formulas | parts 6 – 11 |
| **4. Mining** | How do we find *all* the good rules at once? | `apriori`, `association_rules` | parts 12 – 19 |

A grocery store asks "what sells with what?" A streaming service asks "who gets played with whom?" Same
question, same method: **association rules**. A rule looks like **{J. Cole} → {Kendrick Lamar}**: *playlists
that contain J. Cole tend to contain Kendrick Lamar too.* Our job is to measure how strong such a rule is.

In [ ]:
import numpy as np
import pandas as pd
from plotnine import ggplot, aes, geom_col, coord_flip, labs
from mlxtend.frequent_patterns import apriori, association_rules

---
## 1. Transactions: playlists as baskets

**About the data.** The artists below are real — eight Latin artists and eight hip-hop artists — but the
**playlists are simulated** by the next cell, so that we know what patterns were planted and can check whether
the method finds them. Nothing here is a statement about real listening habits. Each simulated listener
has a taste (Latin, hip-hop, or mixed), picks a few artists accordingly, and Bad Bunny is added to most
playlists regardless of taste, and a handful of pairings are deliberately made more likely — for example,
listeners with J. Cole usually add Kendrick Lamar. *Run the cell;
you do not need to study it.*

In [ ]:
LATIN = ["Bad Bunny", "J Balvin", "Karol G", "Shakira", "Rauw Alejandro", "Peso Pluma", "Daddy Yankee", "Rosalía"]
HIPHOP = ["Kendrick Lamar", "Drake", "J. Cole", "Travis Scott", "Cardi B", "Megan Thee Stallion", "Future", "Tyler, the Creator"]

rng = np.random.default_rng(5544)

def simulate_playlist():
    taste = rng.choice(["latin", "hiphop", "mixed"], p=[0.4, 0.4, 0.2])
    p_latin = {"latin": 0.45, "hiphop": 0.05, "mixed": 0.25}[taste]      # chance of including each Latin artist
    p_hiphop = {"latin": 0.05, "hiphop": 0.45, "mixed": 0.25}[taste]     # ... and each hip-hop artist
    artists = set(a for a in LATIN if rng.random() < p_latin) | set(a for a in HIPHOP if rng.random() < p_hiphop)
    if rng.random() < 0.75:                         artists.add("Bad Bunny")          # Bad Bunny is on most playlists, whatever the taste
    if "J. Cole" in artists and rng.random() < 0.7: artists.add("Kendrick Lamar")    # planted pairings
    if "Karol G" in artists and rng.random() < 0.6: artists.add("Shakira")
    if "Future" in artists and rng.random() < 0.6:  artists.add("Travis Scott")
    if "Cardi B" in artists and rng.random() < 0.6: artists.add("J Balvin")          # a planted cross-genre pairing
    if len(artists) < 2:                                                             # every playlist has at least 2 artists
        artists |= set(rng.choice(LATIN + HIPHOP, size=2, replace=False))
    return sorted(artists)

playlists = [simulate_playlist() for _ in range(400)]
len(playlists), playlists[:3]

`playlists` is a **list of lists**: one inner list per playlist (a *transaction* or *basket*), holding the
artists (*items*) in it. Playlists have different lengths — that is why this is not yet a table.

### From list of lists to one row per (playlist, artist)

The same two moves as PA 7.1: give each playlist an id, then `explode` the list so each artist gets its own row.

In [ ]:
long = (pd.DataFrame({"artist": playlists})
          .assign(playlist_id=lambda d: d.index)
          .____("artist")                     # one row per (playlist, artist)
          .reset_index(drop=True))
print(long.shape)
long.head(8)

### The one-hot table

Association-rule tools want **one row per transaction, one column per item**, with `True`/`False`.
`pd.crosstab` counts (playlist, artist) pairs; `> 0` turns the counts into `True`/`False`.

In [ ]:
basket = pd.____(long["playlist_id"], long["artist"]) > 0
print(basket.shape)
basket.iloc[:5, :6]

✅ **Checkpoint 1.** (a) What does one **row** of `basket` represent? One **column**? (b) `basket.shape` is
(400, 16) — where do the two numbers come from? (c) Why `> 0` rather than keeping the counts?

**Your answer:** *(write it here — replace this line)*

---
## 2. Support: how common is an item?

**Support** of an item = the share of baskets that contain it. Because the column is `True`/`False`, its
**mean** is exactly that share (`True` counts as 1).

In [ ]:
artist_support = basket.____().sort_values(ascending=False)
artist_support.round(3)

In [ ]:
support_df = artist_support.reset_index()
support_df.columns = ["artist", "support"]
support_df["artist"] = pd.Categorical(support_df["artist"], categories=support_df["artist"][::-1])   # keep the order in the plot

(ggplot(support_df, aes(x="artist", y="support"))
 + geom_col()
 + coord_flip()
 + labs(title="Share of playlists containing each artist", x="", y="support"))

Support of a **set** of items = the share of baskets that contain **all** of them. `.all(axis=1)` asks, row by
row, "are all of these columns `True`?"

In [ ]:
both = basket[["J. Cole", "Kendrick Lamar"]].____(axis=1)       # True only when BOTH are on the playlist
both.mean()

✅ **Checkpoint 2.** (a) Which artist has the highest support, and roughly what share of playlists is that?
(b) Why is `basket[["J. Cole", "Kendrick Lamar"]].mean()` (no `.all`) *not* the support of the pair?

**Your answer:** *(write it here — replace this line)*

---
## 3. Confidence and lift: from "together" to "if … then …"

A rule **{A} → {B}** ("A is the *antecedent*, B the *consequent*") is scored with two numbers:

| Measure | Formula | Question it answers |
|---|---|---|
| **confidence** | support(A and B) ÷ support(A) | *Of the baskets with A, what share also have B?* |
| **lift** | support(A and B) ÷ [ support(A) × support(B) ] | *How many times more often than chance do A and B appear together?* |

Lift compares the observed co-occurrence with what independence would predict. **Lift ≈ 1** → no
relationship; **lift > 1** → A makes B more likely; **lift < 1** → A makes B *less* likely.

### Rule 1: {J. Cole} → {Kendrick Lamar}

In [ ]:
s_cole = basket["J. Cole"].mean()
s_kendrick = basket["Kendrick Lamar"].mean()
s_both = basket[["J. Cole", "Kendrick Lamar"]].all(axis=1).mean()

confidence = s_both / ____
lift = s_both / (____ * ____)
print(f"support(J. Cole) = {s_cole:.3f}   support(Kendrick) = {s_kendrick:.3f}   support(both) = {s_both:.3f}")
print(f"confidence = {confidence:.3f}   lift = {lift:.2f}")

### Rule 2: {Drake} → {Bad Bunny}

Same formulas, a very popular consequent:

In [ ]:
s_drake = basket["Drake"].mean()
s_bunny = basket["Bad Bunny"].mean()
s_both2 = basket[["Drake", "Bad Bunny"]].all(axis=1).mean()

print(f"confidence = {s_both2 / s_drake:.3f}   lift = {s_both2 / (____ * ____):.2f}")

✅ **Checkpoint 3.** (a) In one sentence each, interpret the confidence and the lift of {J. Cole} → {Kendrick
Lamar} with the numbers. (b) Rule 2 has a respectable confidence but a lift close to 1. What does that
combination tell a streaming service? (c) Does the rule {Kendrick Lamar} → {J. Cole} have the same confidence?
The same lift? Why?

**Your answer:** *(write it here — replace this line)*

---
## 4. Mining every rule at once: `apriori` and `association_rules`

With 16 artists there are already thousands of possible rules. `apriori` finds every **frequent itemset**
(every set of items whose support is at least `min_support`); `association_rules` then turns those itemsets
into rules and scores them. Two knobs:

- `min_support` — ignore combinations that are too rare to trust;
- a `metric` and `min_threshold` — keep only rules that score well enough (here: confidence ≥ 0.30).

In [ ]:
frequent_itemsets = ____(basket, min_support=0.05, use_colnames=True)
frequent_itemsets["n_items"] = frequent_itemsets["itemsets"].apply(len)
print(frequent_itemsets.shape)
frequent_itemsets.sort_values("support", ascending=False).head(8)

In [ ]:
def make_rules(frequent_itemsets, min_confidence):
    """Association rules from frequent itemsets, keeping those with confidence >= min_confidence."""
    return association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

rules = make_rules(frequent_itemsets, ____)
print(len(rules), "rules")
rules.columns.tolist()

`antecedents` and `consequents` are frozensets — fine for the computer, hard to read. Keep the **simple rules** —
one artist on each side, the easiest to read and to act on — turn the sets into text, and keep the columns we
understand:

In [ ]:
one_each_side = (rules["antecedents"].apply(len) == 1) & (rules["consequents"].apply(len) == 1)
rules = rules[one_each_side]
print(len(rules), "simple rules")
rules["antecedent"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules["consequent"] = rules["consequents"].apply(lambda s: ", ".join(sorted(s)))
show = ["antecedent", "consequent", "support", "confidence", "lift"]

rules.sort_values(____, ascending=False)[show].head(8).round(3)

In [ ]:
rules.sort_values(____, ascending=False)[show].head(8).round(3)

✅ **Checkpoint 4.** (a) What do most of the highest-**confidence** rules have in common, and why is that a
trap? (b) What kinds of pairings rise to the top when you sort by **lift** instead? Do they match what was
planted in the simulation? (c) Find the cross-genre rule involving **Cardi B**. What are its confidence and
lift, and how would you explain it to a non-technical manager?

**Your answer:** *(write it here — replace this line)*

---
## The four lines to keep

| | |
|---|---|
| **Baskets** | list of lists → `explode` → `pd.crosstab(id, item) > 0`: one row per basket, one `True`/`False` column per item |
| **Support** | `basket["A"].mean()` for one item; `basket[["A", "B"]].all(axis=1).mean()` for a set |
| **Confidence & lift** | confidence = supp(A,B) / supp(A); lift = supp(A,B) / (supp(A) · supp(B)); lift ≈ 1 means "no relationship" |
| **Mining** | `apriori(basket, min_support=…, use_colnames=True)` → `association_rules(…, metric="confidence", min_threshold=…)`; **sort by lift, check support** |

PA 7.1 (the groceries data) is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).